In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb
!pip install langchain_text_splitters
!pip install pydantic
!pip install ipywidgets


In [2]:
from langchain_core.documents import Document
sample_doc = Document(
    page_content="Hello world",
    metadata={"source": "https://www.google.com"}
)

In [3]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello world')

In [4]:
type(sample_doc)

langchain_core.documents.base.Document

In [5]:
# text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

C:\Users\New\AppData\Local\Temp\ipykernel_7508\4093581416.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [6]:
document = loader.load()

In [10]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [11]:
# PDF data

from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data/research.pdf")

document = pdf_loader.load()
document

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-06-25T15:05:42+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

In [12]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("data/research.pdf")

document = pdf_loader.load()
document

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': '', 'creationdate': '2026-06-25T15:05:42+00:00', 'source': 'data/research.pdf', 'file_path': 'data/research.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2026-06-25T15:05:42+00:00', 'trapped': '', 'modDate': "D:20260625150542+00'00'", 'creationDate': "D:20260625150542+00'00'", 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Bra

## Ingestion pipeline

In [7]:
#data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## Documents

In [8]:
def load_all_pdfs():
    folder_path= "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [9]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [10]:
type(all_pdf_documents[0])
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [12]:
chunks = split_docs(all_pdf_documents)

In [13]:
len(chunks)

321

## Embedding

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        print("loading model.....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())
        

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [16]:
embedding_manager = EmbeddingManager()

loading model..... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


### Vector Store

In [17]:
import chromadb
import uuid

In [18]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"): 
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        # Create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        # Create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name, 
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )
        print("initialised the vector store with collection=", self.collection_name)
        print("documents in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
            
        ids = []
        all_metadata = []
        documents_content = []  
        embeddings_list = []
        
        # Fixed: changed 'document' to 'documents' to match the argument
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): 
            doc_id = f"doc_{uuid.uuid4()}" 
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content) 
            all_metadata.append(metadata)
            
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist()) 
            
        # Fixed: 'self.colection' -> 'self.collection' AND 'metadata' -> 'metadatas'
        self.collection.add(
            ids=ids, 
            metadatas=all_metadata, 
            documents=documents_content, 
            embeddings=embeddings_list
        )
        print("total documents added in vector store:", len(documents_content))
        print("documents in collection:", self.collection.count())

In [19]:
vector_store = VectorStoreManager()

initialised the vector store with collection= pdf_documents
documents in collection: 1284


In [20]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding= embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (321, 384)
total documents added in vector store: 321
documents in collection: 1605


## Retrieval Pipeline

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store): 
        self.embedding_manager = embedding_manager 
        self.vector_store = vector_store 
        
    def retriever(self, query, top_k=5, score_threshold=0.0): 
        # 1. Generate query embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0] 
        
        # 2. Query the vector store
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()], 
            n_results=top_k
        ) 
        
        retrieved_docs = []  # Fixed: Initialize the list properly
        
        # 3. Process results (Fixed: changed 'document' to 'documents')
        if results.get("documents") and results["documents"][0]: 
            ids = results["ids"][0] 
            metadatas = results["metadatas"][0]  # Fixed typo: metadats -> metadatas
            documents = results["documents"][0] 
            distances = results["distances"][0] 
            
            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)): 
                similarity_score = 1 - distance 
                
                if similarity_score >= score_threshold: 
                    retrieved_docs.append({
                        "id": doc_id, 
                        "document": document, 
                        "metadata": metadata, 
                        "distance": distance, 
                        "similarity_score": similarity_score, 
                        "rank": i + 1 
                    }) 
            
            print(f"retrieved {len(retrieved_docs)} documents:")  # Fixed typo: restricted -> retrieved
        else: 
            print("no document found") 
            
        return retrieved_docs

In [23]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [24]:
results = rag_retriever.retriever("What is encoder decoder?")
print(results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents:
[{'id': 'doc_6965129a-33e7-4a11-9363-6d9223c88a7d', 'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention', 'metadata': {'doc_index': 18, 'language': 'en-US', 'publisher': 'Curran Associates, Inc.', 'book': 'Advances in Neural Information Processing Systems 30', 'title': 'Attention is All you Need', 'content_length': 447, 'published': '2017', 'date': '2017', 'total_pages': 11, 'creationdate': '2026-06-25T15:05:42+00:00', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'type': 'Conference Proceedings', 'lastpage': 

# Integrate RAG with LLMs

### Groq

In [25]:
API_KEY_GROQ = "Enter key"# you will have to enter your own security key

In [26]:
!pip install langchain-groq

In [27]:
from langchain_groq import ChatGroq

llm = llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model="qwen/qwen3-32b",
    temperature=  0.1,#creativity high, fact high usecases ===> temp high
    max_tokens=1024
)

In [28]:
 def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retriever(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    #context + query
    prompt = f""" use given context to generate the answer for the query
                    Context : {context}
                    Query: {query}"""

    #invoke llm => send question => returns response
    response= llm.invoke([prompt.format(context = context, query=query)]) #expecting a string as prompt
    return response.content

In [29]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents:


In [30]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling what I know about RAG. From the context provided, it's mentioned in a survey that reviews state-of-the-art RAG methods. The context also talks about paradigms like naive RAG and mentions an arXiv paper from March 2024.

First, I need to define RAG. RAG stands for Retrieval-Augmented Generation. It's a technique used in natural language processing, right? The main idea is combining retrieval-based methods with generative models. So, when a model needs to answer a question, it first retrieves relevant information from a large corpus and then uses that information to generate a more accurate and informed response.

The context mentions "naive RAG" as one of the paradigms. I should explain that naive RAG might be the basic form where retrieval and generation are done in a straightforward way without much optimization. Then there are more advanced versions that improve upon this, maybe with better retrieval strategie